In [2]:
import pandas as pd
import numpy as np

# 改成你要檢查的檔案
FILE_PATH = r"C:\Users\user\Climate_project\程式作業\Daxue＿village\qm_corrected_output\tmean_corrected_daily_final_models.parquet"

df = pd.read_parquet(FILE_PATH)

df["date"] = pd.to_datetime(df["date"])
df["year"] = df["date"].dt.year
df["scenario"] = df["scenario"].astype(str).str.lower()
df["value"] = pd.to_numeric(df["value"], errors="coerce")

# 去重，避免 historical 那種重複問題
df = df.drop_duplicates(
    subset=["date", "model", "scenario", "variable"]
).copy()

# 只看 future/RCP
fut = df[df["scenario"].isin(["rcp26", "rcp45", "rcp60", "rcp85"])].copy()

# =========================
# 1. 整段期間 mean / p90 / p95
# =========================
summary = (
    fut.groupby("scenario")["value"]
       .agg(
           mean="mean",
           p90=lambda x: x.quantile(0.9),
           p95=lambda x: x.quantile(0.95),
           min="min",
           max="max",
           n="count"
       )
       .reset_index()
)

print("=== RCP overall temperature summary ===")
print(summary)


# =========================
# 2. 2034 年檢查
# =========================
annual = (
    fut.groupby(["scenario", "year"], as_index=False)["value"]
       .mean()
       .rename(columns={"value": "annual_mean_temp"})
)

print("\n=== 2034 annual mean temperature ===")
print(annual[annual["year"] == 2034].sort_values("annual_mean_temp", ascending=False))


# =========================
# 3. 各 RCP 年平均趨勢 slope
# =========================
slope_rows = []

for s in ["rcp26", "rcp45", "rcp60", "rcp85"]:
    sub = annual[annual["scenario"] == s].sort_values("year")

    if len(sub) >= 2:
        slope, intercept = np.polyfit(sub["year"], sub["annual_mean_temp"], 1)
    else:
        slope = np.nan

    slope_rows.append({
        "scenario": s,
        "slope_temp_per_year": slope
    })

slope_df = pd.DataFrame(slope_rows)

print("\n=== Annual mean temperature slope ===")
print(slope_df.sort_values("slope_temp_per_year", ascending=False))


# =========================
# 4. 共同模式檢查
# =========================
common_models = set.intersection(
    *[
        set(fut[fut["scenario"] == s]["model"].unique())
        for s in ["rcp26", "rcp45", "rcp60", "rcp85"]
    ]
)

print("\n=== Common models ===")
print("n common models:", len(common_models))
print(sorted(common_models))

fut_common = fut[fut["model"].isin(common_models)].copy()

common_summary = (
    fut_common.groupby("scenario")["value"]
       .agg(
           mean="mean",
           p90=lambda x: x.quantile(0.9),
           p95=lambda x: x.quantile(0.95),
           n_models=lambda x: fut_common.loc[x.index, "model"].nunique()
       )
       .reset_index()
)

print("\n=== Common-model RCP summary ===")
print(common_summary)


# =========================
# 5. 共同模式 2034 年
# =========================
annual_common = (
    fut_common.groupby(["scenario", "year"], as_index=False)["value"]
       .mean()
       .rename(columns={"value": "annual_mean_temp"})
)

print("\n=== Common-model 2034 annual mean temperature ===")
print(
    annual_common[annual_common["year"] == 2034]
    .sort_values("annual_mean_temp", ascending=False)
)


# =========================
# 6. 快速判斷提示
# =========================
print("\n=== Quick interpretation guide ===")
print("如果 overall/common-model 都是 rcp26 > rcp85，可能是校正後資料或模式集合造成排序反轉。")
print("如果 common-model 後排序變正常，主要是不同 RCP 模式集合不一致造成。")
print("如果排序仍反轉，可能是校正方法或輸出 parquet 還是舊版本。")

=== RCP overall temperature summary ===
  scenario       mean        p90        p95       min        max       n
0    rcp26  23.219028  29.909087  30.516090  1.700962  34.486067  226300
1    rcp45  23.055850  29.882076  30.508777  0.959679  34.620567  305505
2    rcp60  23.012460  29.777905  30.377825  2.482079  34.589367  169725
3    rcp85  23.003144  29.839143  30.406928  1.432579  34.123567  339450

=== 2034 annual mean temperature ===
    scenario  year  annual_mean_temp
28     rcp26  2034         23.617807
59     rcp45  2034         23.486044
90     rcp60  2034         23.336298
121    rcp85  2034         23.278464

=== Annual mean temperature slope ===
  scenario  slope_temp_per_year
0    rcp26             0.036603
1    rcp45             0.029196
3    rcp85             0.021957
2    rcp60             0.016437

=== Common models ===
n common models: 13
['CCSM4', 'CESM1-CAM5', 'CSIRO-Mk3-6-0', 'GFDL-ESM2G', 'IPSL-CM5A-LR', 'IPSL-CM5A-MR', 'MIROC-ESM', 'MIROC-ESM-CHEM', 'MIROC5', 'M